# 118 — Memoria, contexto y continuidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

El LLM es una función sin estado: solo "recuerda" su ventana de contexto. La
continuidad se construye con tres almacenes:

- **Contexto (memoria de trabajo):** instrucciones + plan + ternas recientes. Volátil,
  cara (se paga por token en CADA llamada), limitada.
- **Memoria persistente:** *episódica* (qué pasó: trazas) y *semántica* (hechos
  destilados con procedencia). Entra al contexto por recuperación selectiva.
- **Checkpoint:** instantánea del estado del bucle (plan + variables + log de efectos
  aplicados) en un punto consistente; su contrato es reanudar SIN repetir efectos.

### 📉 Gestionar el contexto que crece

Estrategias: **compactar** (ternas viejas → resumen estructurado; con pérdida — errores
y decisiones se conservan textuales), **externalizar** (detalle → archivo/BD + puntero
en contexto) y **seleccionar** (recuperar solo lo relevante al paso actual, parte 08).

```text
contexto = instrucciones (fijo) + plan (siempre visible)
         + resumen de lo hecho + últimas K ternas + recuperado bajo demanda
```

Riesgo propio de la memoria: lo persistido vuelve a entrar en sesiones futuras — un
dato envenenado hoy es una "verdad recordada" mañana. Etiquetar procedencia y curar
(caducidad, corrección) no es opcional.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Diseña el checkpoint.** Ejecuta `run_lab("agent", seed=118)` y construye
a mano el checkpoint que tomarías DESPUÉS del paso 1 (tras observar `status`): incluye
plan con estados, variables verificadas y log de efectos. Demuestra que al reanudar
desde él, el agente NO repite `status()` y sí ejecuta `sum`.

**Ejercicio 2 — Presupuesto de contexto a mano.** Instrucciones 900 tokens, plan 350,
cada terna 280, ventana útil 6.000. (a) ¿En qué paso desborda un agente que lo conserva
todo? (b) Con la regla "resumen 500 + últimas 4 ternas textuales", ¿cuál es el contexto
estable por llamada y cuántos tokens pagas acumulados en 30 pasos en cada variante
(aprox.)?

**Ejercicio 3 — Clasifica los recuerdos.** Decide destino (contexto / episódica /
semántica / no persistir) y procedencia (usuario / tool / deducción) para: (a) "el
usuario pidió salida en CSV"; (b) la traza completa de la migración de ayer; (c) "el
endpoint /v2 responde 40 % más rápido" medido por una tool; (d) el token de API usado
en la sesión; (e) "creo que el módulo X es el frágil" (hipótesis del modelo).

**Ejercicio 4 — Compacta una traza.** Toma la traza del laboratorio y escríbela como
resumen de compactación estructurado (qué se hizo, qué quedó verificado, qué falta)
en ≤ 40 palabras, conservando textual lo imprescindible. ¿Qué información sacrificaste
y qué consecuencia tendría si la tarea continuara 20 pasos más?

In [ ]:
# TODO: ejecuta run_lab("agent", seed=118)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 1: checkpoint tras el paso 1
result = run_lab("agent", seed=118)
trace = result["result"]["trace"]
checkpoint_1 = {
    "plan": [
        # {"subtarea": "verificar estado", "estado": "?"},
        # {"subtarea": "sumar 7+5", "estado": "?"},
    ],
    "verificado": {},      # hechos anclados a observaciones hasta el paso 1
    "log_efectos": [],     # acciones ya aplicadas (no repetir al reanudar)
}
# simula la reanudación: ¿qué única acción queda pendiente?


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 2: presupuesto de contexto
INSTRUCCIONES, PLAN, TERNA, VENTANA = 900, 350, 280, 6000
# (a) paso en el que desborda si se conserva todo:
paso_desborde = None
# (b) contexto estable con resumen 500 + 4 ternas textuales:
contexto_estable = None
# tokens de entrada acumulados en 30 pasos, ambas variantes (aprox):
acumulado_sin_gestion = None
acumulado_con_gestion = None


## Reflexión

1. El laboratorio termina en dos pasos y no necesita checkpoint. ¿En qué momento exacto
   de una tarea de 40 pasos tomarías checkpoints y por qué "nunca en mitad de un
   efecto"?
2. La compactación es una operación con pérdida. ¿Qué dos tipos de contenido deben
   conservarse textuales aunque todo lo demás se resuma, y qué error induce omitirlos?
3. ¿Por qué la memoria semántica recuperada debe tratarse como dato de baja procedencia
   aunque la haya escrito el propio agente?